In [1]:
import matplotlib.pyplot as plt
import sys
import os
import importlib

# Add the parent directory to the path to import src as a package
sys.path.insert(0, os.path.abspath('..'))
from src import dataloader

importlib.reload(dataloader)
from src import utils
from src import export

importlib.reload(export)
from src.export import write_dyad_to_uniwaw_imported

%matplotlib widget 
plot_flag = False

# Example of saving data for multiple dyads to NCDF while creating the folder structure

In [2]:
write_dyad_to_uniwaw_imported(['W_030', 'W_003'], eeg_filter_type='iir', time_margin=10, verbose=False)

Detected events: [{'name': 'Brave', 'start': 387.806640625, 'duration': 59.3310546875}, {'name': 'Peppa', 'start': 248.5107421875, 'duration': 59.6328125}, {'name': 'Incredibles', 'start': 318.3603515625, 'duration': 59.212890625}, {'name': 'Talk_1', 'start': 594.4892578125, 'duration': 181.0556640625}, {'name': 'Talk_2', 'start': 836.7275390625, 'duration': 181.056640625}]
Applying iir filters to EEG data.
Reseting the EEG time to the start of Peppa
ET time range: 241.59s to 461.89s
Events from ET annotations:
[None 'Peppa' 'Incredibles' 'Brave']
Reseting the ET time to the start of Peppa
Processing member: ch, blink column: ET_ch_blinks
Processing member: cg, blink column: ET_cg_blinks
Column ET_ch_blinks contains NaN values, applying forward fill before decimation.
Column ET_cg_blinks contains NaN values, applying forward fill before decimation.
Event Peppa start times are consistent within 0.0 seconds.
Event Incredibles differ in start times by: abs(0.0078125) seconds.
Event Brave 

# Example of reading from ncdf file to xarray

In [3]:
### Load one exported `.nc` file to xarray (`EEG`, `ch`, `Peppa`)

from pprint import pprint
from src.export import load_xarray_from_netcdf, get_export_metadata

# Selection
dyad_id = "W_030"
selected_modality = "EEG"
selected_member = "ch"
selected_event = "Peppa"

member_folder = {"ch": "child", "cg": "caregiver"}[selected_member]

nc_path = os.path.join("../data/UNIWAW_imported", selected_modality, dyad_id, member_folder, 
    f"{dyad_id}_{selected_modality}_{selected_member}_{selected_event}.nc"
)

# Load to xarray
data_xr = load_xarray_from_netcdf(str(nc_path))

# Optional: read structured metadata payload
metadata = get_export_metadata(data_xr)

print(f"Loaded: {nc_path}")
print(data_xr)
print("Metadata keys:", list(metadata.keys()))
pprint(metadata, sort_dicts=False)


Loaded: ../data/UNIWAW_imported/EEG/W_030/child/W_030_EEG_ch_Peppa.nc
<xarray.DataArray 'W_030 EEG ch Peppa with ±10s margin' (time: 10193,
                                                         channel: 21)> Size: 2MB
array([[-97.45345145, -86.96672122, -39.96571836, ...,  -7.46349458,
         -3.64278855,   4.18598785],
       [-63.83729271, -51.04183786,  -7.15739937, ...,   4.03579056,
         -0.9017896 ,  10.2900968 ],
       [-44.46871673, -26.97992007,   9.66799152, ...,   9.56932999,
          0.55484251,  14.33467554],
       ...,
       [ 35.87007552,  16.18397873,  99.19952674, ...,  36.33018111,
          6.54528401,  28.1131186 ],
       [ 31.42516263,  14.15968767,  86.91312848, ...,   8.78991483,
         14.57440476,  19.36983445],
       [ -6.84919819, -19.11857816,  56.62971498, ..., -50.31087026,
         10.75834846,  13.97787912]], shape=(10193, 21))
Coordinates:
  * time     (time) float64 82kB -10.0 -9.992 -9.984 ... 69.61 69.62 69.62
  * channel  (channel) 

In [4]:
metadata = get_export_metadata(data_xr)
notch_q = metadata['eeg']['filtration']['notch']['Q']
print(f"Notch Q: {notch_q}")

Notch Q: 30
